# Hill Cipher Encryption
1. Let's first set the alphabet in numerical order using a dictionary.

In [ ]:
alphabet = {chr(ord('A') + i): i for i in range(26)}
for key in alphabet:
    print(key, alphabet[key])

A 0
B 1
C 2
D 3
E 4
F 5
G 6
H 7
I 8
J 9
K 10
L 11
M 12
N 13
O 14
P 15
Q 16
R 17
S 18
T 19
U 20
V 21
W 22
X 23
Y 24
Z 25


2. Next is a key. Using the formula Cipher = Key * Plaintext (mod 26), we can encrypt our plain text with a n x n matrix, where n >= 2 and it has the gcd(det, 26) = 1.
So we need:
find_det(key)

Using gaussian elimination method.

In [40]:
def find_det(matrix):
    # Get the size of the matrix (n x n)
    n = len(matrix)
    
    # Make a copy of the matrix so we do not alter the original data
    mat = [row[:] for row in matrix]
    
    det_sign = 1  # Tracks sign changes due to row swaps

    for i in range(n):  
        # Find a pivot in the current column
        pivot_row = i
        
        # If all numbers in this column are zero, the determinant is 0
        while pivot_row < n and mat[pivot_row][i] == 0:
            pivot_row += 1
            
        if pivot_row == n:
            return 0

        # If the pivot is not in the current row, swap rows
        if pivot_row != i:
            mat[i], mat[pivot_row] = mat[pivot_row], mat[i]
            det_sign *= -1  # Each row swap flips the sign of the determinant

        # Eliminate all values below the pivot
        for j in range(i + 1, n):
            # Calculate the multiplier to make the entry zero
            factor = mat[j][i] / mat[i][i]
            for k in range(i, n):
                mat[j][k] -= factor * mat[i][k]

    # Multiply the numbers along the main diagonal
    determinant = det_sign
    for i in range(n):
        determinant *= mat[i][i]

    return determinant

# Example usage with a 3x3 matrix
key = [
    [3, 3],
    [2, 5]
]

det = find_det(key)
print("Determinant:", det)

Determinant: 9.0


In [41]:

def find_gcd(num1, det):
    num = num1
    denom = det
    r = num % denom
    
    while r != 0:
        num, denom = denom, r
        r = num % denom
        
    return denom;
            
        
print("GCD:", find_gcd(26, det))
        

GCD: 1.0


3. Divide the plaintext into n x len(plaintext) matrix. Transpose the matrix turning it into a row vector for ease of iteration

In [42]:
def block_plaintext(plaintext, n):
    if n <= 0:
        raise ValueError("n must be positive")

    cleaned = ''.join(
        char for char in plaintext.upper()
        if char in alphabet
    )

    # Pad the final block with X
    padding = (-len(cleaned)) % n
    cleaned += "X" * padding

    return [
        [alphabet[char] for char in cleaned[i:i + n]]
        for i in range(0, len(cleaned), n)
    ]



plaintext = "hello"
blocks = block_plaintext(plaintext, 2)
print(blocks)
vector_blocks = [[[val] for val in block] for block in blocks]
print(vector_blocks)

[[7, 4], [11, 11], [14, 23]]
[[[7], [4]], [[11], [11]], [[14], [23]]]


4. We multiply using our key matrix as a multiple to our blocked plaintext then finding the mod 26 of the product. 

In [ ]:
def encrypt_column_block(col_vector, key_matrix):
    """
    Multiplies a square key matrix by a column vector block (K x P) 
    and applies modulo 26.
    """
    n = len(key_matrix)
    # Initialize an empty column vector for the result
    result = [[0] for _ in range(n)]
    
    for i in range(n):
        row_sum = 0
        for k in range(n):
            # Key row i, column k multiplied by column vector row k, column 0
            row_sum += key_matrix[i][k] * col_vector[k][0]
            
        result[i][0] = row_sum % 26
        
    return result


encrypted_blocks = [encrypt_column_block(b, key) for b in vector_blocks]
reversed_alphabet_dict = {v: k for k, v in alphabet.items()}  



print("Encrypted block values:", encrypted_blocks)

Encrypted block values: [[[7], [8]], [[14], [25]], [[7], [13]]]
